# 07 — Async: two tools at once

In module 02, when the model asked for two tools in one reply, we ran them **one after another**. That was correct, and slow on purpose.

Every modern agent SDK is **async-first**. `Runner.run` in the OpenAI Agents SDK, LangGraph nodes, CrewAI kicks — they all `await`. If the first time you see `await` is inside LangGraph, you will copy the line and not know why it is there.

Today we walk slowly: a normal `def`, the same function as `async def`, why the call is a coroutine and not the result, then `await`, then `gather`. After that, the same two travel tools, timed sequential vs together. Then the official loop dispatches a pair of `tool_calls` with `asyncio.gather`.

The lookups themselves are instant CSVs. We add a one-second wait in each tool so the clock can teach. That wait stands in for a database, HTTP, or MCP call.

Module 08 is the OpenAI Agents SDK for real. It will `await`. This module is why.


## 1. Learn

```
02  two tool_calls, we ran them in a for-loop (sequential)
03  the loop, plus await Runner.run from 02
06  those tools can live in another process
07  you are here — when two tools are independent, run them together
08  OpenAI Agents SDK (await Runner.run)
09  LangGraph
```

You already typed `await Runner.run` in 02. That line worked because Jupyter has an event loop. Today we open the word.

A normal function (`def`) runs from the first line to the `return`. The caller gets the **result**.

An `async def` function is the same idea with one extra power: it can **pause** and let other work run. Calling it does **not** run the body. It hands you a **coroutine** — a ticket for work that has not started. `await` is how you cash the ticket and get the result.

`asyncio.gather` is how you cash **several** tickets at once. The clock is then the **slowest** one, not the sum.

```
sequential                    gather

await flight   1s             flight  ──┐
await fact     1s             fact    ──┼──  ~1s  (the slower one)
              ---                       ┘
               2s
```

Only do this when the calls do not depend on each other. Barcelona → Amman, *then* a fact about the landing city, is still sequential. A flight **and** a fact about Madrid (you already know the city) can be together.

The lookups themselves are instant CSVs. We add a one-second wait so the clock can teach. That wait stands in for a database, HTTP, or MCP call.


## 2. Do

### Load the environment and the two CSV tools


In [1]:
from pathlib import Path
import asyncio
import csv
import json
import os
import time

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing."

client = OpenAI()
facts_rows = list(csv.DictReader(open(ROOT / "data" / "fun_facts.csv")))
flight_rows = list(csv.DictReader(open(ROOT / "data" / "flight_data.csv")))
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano


### Primer — from `def` to `async def`

Start with a function you would have written in module 00. No network. No sleep. Call it, get a string.


In [2]:
def greet(name):
    return "hello " + name


print(greet("Sam"))
print("type of the return:", type(greet("Sam")).__name__)


hello Sam
type of the return: str


Same body. Two extra letters: `async` before `def`. That is now a **coroutine function**.

Call it the same way. Look at what comes back. Do not `await` yet.


In [3]:
async def greet_async(name):
    return "hello " + name


ticket = greet_async("Sam")
print(ticket)
print("type of the return:", type(ticket).__name__)


<coroutine object greet_async at 0x10c3b5cc0>
type of the return: coroutine


That is a **coroutine**, not `"hello Sam"`. The body has not run. Python is warning you (or will, when the cell finishes) that the ticket was never awaited. That warning is the lesson.

`await` cashes the ticket. The body runs. You get the string.

In this notebook you can `await` in a cell. Jupyter *is* the event loop. You do not need `asyncio.run` here. In a `.py` file you would wrap the top call in `asyncio.run(...)`.


In [4]:
result = await greet_async("Sam")
print(result)
print("type of the return:", type(result).__name__)


hello Sam
type of the return: str


### Why change the function at all?

A normal `def` that waits — `time.sleep`, a slow HTTP call, a database — **blocks everything**. The program sits on that line. Nothing else can start.

`await asyncio.sleep(...)` is a wait that **lets go**. The loop is free to run other awaited work during that second.

Watch the prints. `start`, then a pause, then `done`, then the return value.


In [5]:
async def pause(name, seconds):
    print("start", name)
    await asyncio.sleep(seconds)
    print("done ", name)
    return name


print("returned:", await pause("alone", 0.3))


start alone


done  alone
returned: alone


### Two waits, one after the other

`await` one, then `await` the other. The second does not start until the first is done. Two one-second waits cost about **two** seconds.

This is the `for` from module 02.


In [6]:
t0 = time.perf_counter()
first = await pause("a", 1)
second = await pause("b", 1)
print("results:", first, second)
print("sequential seconds:", round(time.perf_counter() - t0, 2))


start a


done  a
start b


done  b
results: a b
sequential seconds: 2.0


You should have seen:

```
start a
done  a
start b
done  b
sequential seconds: ~2
```

### `gather` — start both, wait for the slower one

`asyncio.gather(...)` takes **coroutines** (the tickets), not already-awaited results.

```
# wrong: you already waited for each one, in order
await asyncio.gather(await pause("a", 1), await pause("b", 1))

# right: hand gather the tickets; it starts them together
await asyncio.gather(pause("a", 1), pause("b", 1))
```

It returns a list of results, **in the same order as the arguments**, when the last one finishes. The clock is about **one** second, not two: both waits overlap.


In [7]:
t0 = time.perf_counter()
together = await asyncio.gather(pause("a", 1), pause("b", 1))
print("results:", together)
print("gather seconds:", round(time.perf_counter() - t0, 2))


start a
start b


done  a
done  b
results: ['a', 'b']
gather seconds: 1.0


You should have seen both `start`s before both `done`s:

```
start a
start b
done  a
done  b
results: ['a', 'b']
gather seconds: ~1
```

`['a', 'b']` — same order you passed them, even if `b` had finished first.

If the two times are still ~2, you used `time.sleep` inside `async def`, or you `await`ed each call *before* giving it to `gather`. The wait must be `await asyncio.sleep`.


### The travel tools — same transformation

`get_fact` and `get_flight` from module 02 are ordinary `def`s. We give them a fake one-second network wait (`time.sleep`) so a sequential pair costs ~2s.

Then we write the **async** twins: `async def` and `await asyncio.sleep`. Same CSV, same return string. The only change that matters is the wait that can let go.

Do **not** put `time.sleep` inside `async def`. That still blocks the loop. `gather` would look concurrent and still take two seconds.


In [8]:
def get_fact(city):
    time.sleep(1)
    needle = city.strip().lower()
    for row in facts_rows:
        if row["City"].lower() == needle:
            return row["Fun Fact"]
    return "no fact for that city"


def get_flight(from_city, to_city):
    time.sleep(1)
    a = from_city.strip().lower()
    b = to_city.strip().lower()
    found = []
    for row in flight_rows:
        if row["from_city"].lower() == a and row["to_city"].lower() == b:
            found.append(row["price"] + " dollars, " + row["duration"] + " minutes")
    return "; ".join(found) if found else "no flight found"


async def get_fact_async(city):
    await asyncio.sleep(1)
    needle = city.strip().lower()
    for row in facts_rows:
        if row["City"].lower() == needle:
            return row["Fun Fact"]
    return "no fact for that city"


async def get_flight_async(from_city, to_city):
    await asyncio.sleep(1)
    a = from_city.strip().lower()
    b = to_city.strip().lower()
    found = []
    for row in flight_rows:
        if row["from_city"].lower() == a and row["to_city"].lower() == b:
            found.append(row["price"] + " dollars, " + row["duration"] + " minutes")
    return "; ".join(found) if found else "no flight found"


### Same two lookups, two clocks

Sydney → Madrid, and a fact about Madrid. The city is known. They do not depend on each other.

First cell: the ordinary `def`s, one after the other. Two `time.sleep(1)` — about **2** seconds.

Second cell: the `async` twins, handed to `gather` as tickets. `flight, fact = await asyncio.gather(...)` unpacks the list of results in argument order. About **1** second.


In [9]:
t0 = time.perf_counter()
flight = get_flight("Sydney", "Madrid")
fact = get_fact("Madrid")
print(flight)
print(fact)
print("sequential seconds:", round(time.perf_counter() - t0, 2))


249.66 dollars, 99 minutes
Madrid is home to the oldest restaurant in the world, Sobrino de Botín.
sequential seconds: 2.01


In [10]:
t0 = time.perf_counter()
flight, fact = await asyncio.gather(
    get_flight_async("Sydney", "Madrid"),
    get_fact_async("Madrid"),
)
print(flight)
print(fact)
print("gather seconds:", round(time.perf_counter() - t0, 2))


249.66 dollars, 99 minutes
Madrid is home to the oldest restaurant in the world, Sobrino de Botín.
gather seconds: 1.0


About two seconds, then about one. That is the only number this module needs to land.

### The official loop, concurrent dispatch

When one model reply has **two** `tool_calls` that do not depend on each other, do not `await` them in a `for`. Build a list of tickets (`run_one(c)` — notice: no `await` on that line) and `gather` the list.

```
results = await asyncio.gather(*[run_one(c) for c in message.tool_calls])
```

The `*` spreads the list into arguments. If the model asked for one tool, `gather` of one item is fine — it just is not faster.

Question: a flight from Sydney to Madrid **and** a fun fact about Madrid.


In [11]:
def schema(tool_name, description, **props):
    return {"type": "function", "function": {
        "name": tool_name, "description": description,
        "parameters": {"type": "object", "properties": props, "required": list(props)}}}

tools = [
    schema("get_fact", "Fun fact about a city.", city={"type": "string"}),
    schema("get_flight", "Flight between two cities.", from_city={"type": "string"}, to_city={"type": "string"}),
]


In [12]:
async def run_one(call):
    args = json.loads(call.function.arguments or "{}")
    if call.function.name == "get_fact":
        return await get_fact_async(args.get("city", ""))
    if call.function.name == "get_flight":
        return await get_flight_async(args.get("from_city", ""), args.get("to_city", ""))
    return "unknown tool"


messages = [
    {
        "role": "system",
        "content": "Use the tools. Do not invent prices or facts. You may call more than one tool.",
    },
    {
        "role": "user",
        "content": "How much is a flight from Sydney to Madrid, and give me a fun fact about Madrid.",
    },
]

for turn in range(4):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=128,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    n = len(message.tool_calls or [])
    print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "n=", n, "---")
    if not message.tool_calls:
        print(message.content)
        break
    messages.append(message)
    t0 = time.perf_counter()
    results = await asyncio.gather(*[run_one(c) for c in message.tool_calls])
    print("dispatch seconds:", round(time.perf_counter() - t0, 2))
    for call, result in zip(message.tool_calls, results):
        print(call.function.name, "->", result[:80])
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})


--- turn 1 finish_reason: tool_calls n= 1 ---


dispatch seconds: 1.0
get_flight -> 249.66 dollars, 99 minutes


--- turn 2 finish_reason: tool_calls n= 1 ---


dispatch seconds: 1.0
get_fact -> Madrid is home to the oldest restaurant in the world, Sobrino de Botín.


--- turn 3 finish_reason: stop n= 0 ---
A flight from **Sydney to Madrid** is **$249.66** and takes about **99 minutes**.

**Fun fact about Madrid:** It’s home to the world’s oldest restaurant, **Sobrino de Botín**.


If turn 1 asked for **both** tools, `dispatch seconds` should be about 1, not 2. If it asked for one, then one, that is sequential by the model's choice — still legal. The landing city question from module 03 must stay sequential. Gather is for independent work.

That `await asyncio.gather` is what the SDK is doing inside `Runner.run` when it sees two tool calls.


## 3. Observe

Write the two clocks down. Sequential should be near 2. Gather should be near 1. If they are the same, the sleeps were not awaited — look at `async def` vs `time.sleep`.


## 4. Challenge

Time these two yourself, with the async travel tools:

- Istanbul fact **and** Tokyo → Moscow flight (independent — can gather)

Bind `seq_seconds` and `gather_seconds`. The check only asks that gather is faster. It does not score the strings.


In [ ]:
# seq_seconds, gather_seconds = ...


In [ ]:
assert gather_seconds < seq_seconds, "gather should beat one-after-another when the work is independent"
print("seq_seconds:   ", seq_seconds)
print("gather_seconds:", gather_seconds)
